In [1]:
!pip install torch --index-url https://download.pytorch.org/whl/cpu
!pip install sentence-transformers scikit-learn

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 MB 5.8 MB/s  0:00:33 eta 0:00:010:00:01
Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 2.1 MB/s  0:00:03 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 2.0 MB/s  0:00:00m ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [torch]m━━━━ 7/8 [torch]]afe]
  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached safeten

In [2]:
products = [
    {
        "id": "lego_castle_001",
        "name": "Lego Castle",
        "description": "Build a medieval castle using interlocking bricks and blocks."
    },
    {
        "id": "lego_city_002",
        "name": "Lego City",
        "description": "Construction building set for creating cities, roads, and structures."
    },
    {
        "id": "wooden_blocks_003",
        "name": "Wooden Blocks",
        "description": "Stackable blocks for construction and creative play."
    },
    {
        "id": "teddy_bear_010",
        "name": "Teddy Bear",
        "description": "Soft plush toy for comfort and cuddling."
    },
    {
        "id": "action_figure_015",
        "name": "Action Figure",
        "description": "Poseable superhero toy for role play."
    }
]

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

for p in products:
    text = p["name"] + " " + p["description"]
    p["embedding"] = model.encode(text)

/home/shivansh/Desktop/OpenSource/interneers-lab/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10269.40it/s]


In [4]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def semantic_search(query, products, model, top_k=3):
    query_emb = model.encode(query)

    scored = []

    for p in products:
        score = cosine_similarity(query_emb, p["embedding"])
        scored.append((p["id"], score))

    scored.sort(key=lambda x: x[1], reverse=True)

    return scored[:top_k]

In [5]:
semantic_search("construction toys", products, model)

[('lego_city_002', np.float32(0.53300637)),
 ('wooden_blocks_003', np.float32(0.50203025)),
 ('teddy_bear_010', np.float32(0.5017936))]

In [6]:
SEARCH_TEST_CASES = [
    {
        "query": "construction toys",
        "relevant_products": [
            "lego_castle_001",
            "lego_city_002",
            "wooden_blocks_003"
        ],
        "irrelevant_products": [
            "teddy_bear_010",
            "action_figure_015"
        ]
    },
    {
        "query": "soft toys for kids",
        "relevant_products": [
            "teddy_bear_010"
        ],
        "irrelevant_products": [
            "lego_castle_001",
            "action_figure_015"
        ]
    }
]

In [7]:
def evaluate_search(products, model, test_cases, k=3):
    total_precision = 0

    for test in test_cases:
        query = test["query"]
        relevant = set(test["relevant_products"])

        results = semantic_search(query, products, model, top_k=k)
        retrieved_ids = [r[0] for r in results]

        hits = sum([1 for r in retrieved_ids if r in relevant])
        precision = hits / k

        total_precision += precision

        print(f"\nQuery: {query}")
        print("Retrieved:", retrieved_ids)
        print("Relevant:", relevant)
        print(f"Precision@{k}: {precision:.2f}")

    avg_precision = total_precision / len(test_cases)

    print("\n🔥 Average Precision:", avg_precision)

In [8]:
evaluate_search(products, model, SEARCH_TEST_CASES)


Query: construction toys
Retrieved: ['lego_city_002', 'wooden_blocks_003', 'teddy_bear_010']
Relevant: {'wooden_blocks_003', 'lego_city_002', 'lego_castle_001'}
Precision@3: 0.67

Query: soft toys for kids
Retrieved: ['teddy_bear_010', 'action_figure_015', 'wooden_blocks_003']
Relevant: {'teddy_bear_010'}
Precision@3: 0.33

🔥 Average Precision: 0.5
